# Spotify Feature Launch Experiment
## Measuring the Impact of an AI Playlist Feature on Premium Conversion

This notebook presents a full A/B testing analysis of a simulated Spotify feature launch experiment. 
The experiment measures whether a new AI-powered playlist feature drives free-to-paid Premium conversions 
across a three-stage funnel: feature adoption, notification re-engagement, and Premium conversion.

**Business Question:** Does the AI playlist feature drive free-to-paid conversion among Spotify users?

**Experiment Design:**
- 10,000 simulated free-tier Spotify users
- 50/50 random assignment to control and treatment groups
- Three-stage funnel analysis with frequentist and Bayesian statistical methods

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import plotly.express as px
from src.simulate import run_simulation
from src.stats_engine import run_analysis

df = run_simulation()
results = run_analysis()

df.head(10)

## Data Overview

The simulation models 10,000 free-tier Spotify users randomly assigned to control and treatment groups 
with an equal 50/50 probability. Each user passes through a three-stage funnel, with each stage modeled 
as an independent Bernoulli trial with group-specific rates.

In [3]:
print(f"Total users: {len(df):,}")
print(f"Control group: {len(df[df['group'] == 'control']):,}")
print(f"Treatment group: {len(df[df['group'] == 'treatment']):,}")
print(f"\nFunnel Summary:")
print(f"  Adopted:    {df['adopted'].sum():,} ({df['adopted'].mean():.1%})")
print(f"  Reengaged:  {df['reengaged'].sum():,} ({df['reengaged'].mean():.1%})")
print(f"  Converted:  {df['converted'].sum():,} ({df['converted'].mean():.1%})")

Total users: 10,000
Control group: 5,076
Treatment group: 4,924

Funnel Summary:
  Adopted:    1,535 (15.3%)
  Reengaged:  1,087 (10.9%)
  Converted:  230 (2.3%)


## Stage 1: Feature Adoption

**Hypothesis:** The AI playlist feature drives higher adoption in the treatment group than control.

A chi-square test of independence was applied to a 2x2 contingency table of adoption outcomes 
across groups. The Wilson method was used to construct a 95% confidence interval around the 
observed lift.

In [4]:
adoption = results["adoption"]

print(f"Control adoption rate:    {adoption['control_rate']:.1%}")
print(f"Treatment adoption rate:  {adoption['treatment_rate']:.1%}")
print(f"Observed lift:            {adoption['observed_lift']:.1%}")
print(f"95% CI on lift:           [{adoption['ci_low']:.1%}, {adoption['ci_high']:.1%}]")
print(f"Chi-square statistic:     {adoption['chi2_statistic']}")
print(f"P-value:                  {adoption['p_value']}")
print(f"Significant:              {adoption['significant']}")
print(f"P(treatment > control):   {adoption['prob_treatment_wins']:.1%}")

Control adoption rate:    5.3%
Treatment adoption rate:  25.7%
Observed lift:            20.4%
95% CI on lift:           [18.5%, 22.2%]
Chi-square statistic:     796.6919
P-value:                  0.0
Significant:              True
P(treatment > control):   100.0%


### Findings

The treatment group adopted the AI playlist feature at a rate of 25.7%, compared to 5.3% in control — 
a 20.4 percentage point lift. The chi-square test returns a p-value of effectively zero, and the 
95% confidence interval of [18.5%, 22.2%] confirms the lift is both statistically significant and 
practically meaningful. The Bayesian analysis assigns a 100% probability that the treatment rate 
genuinely exceeds control, leaving no ambiguity about the direction of the effect.

## Stage 2: Notification Re-engagement

**Hypothesis:** A personalized push notification drives higher re-engagement among non-adopters 
in the treatment group than a generic notification in control.

Only users who did not adopt the feature in Stage 1 were eligible for re-engagement. The same 
chi-square and Bayesian framework was applied to this subset.

In [5]:
reengagement = results["reengagement"]

print(f"Control re-engagement rate:    {reengagement['control_rate']:.1%}")
print(f"Treatment re-engagement rate:  {reengagement['treatment_rate']:.1%}")
print(f"Observed lift:                 {reengagement['observed_lift']:.1%}")
print(f"95% CI on lift:                [{reengagement['ci_low']:.1%}, {reengagement['ci_high']:.1%}]")
print(f"Chi-square statistic:          {reengagement['chi2_statistic']}")
print(f"P-value:                       {reengagement['p_value']}")
print(f"Significant:                   {reengagement['significant']}")
print(f"P(treatment > control):        {reengagement['prob_treatment_wins']:.1%}")

Control re-engagement rate:    7.5%
Treatment re-engagement rate:  14.3%
Observed lift:                 6.8%
95% CI on lift:                [5.1%, 8.5%]
Chi-square statistic:          119.7122
P-value:                       0.0
Significant:                   True
P(treatment > control):        100.0%


### Findings

Among non-adopters, the personalized notification re-engaged 14.3% of treatment users compared to 
7.5% in control — a 6.8 percentage point lift. While the effect size is smaller than Stage 1, it 
remains highly significant with a confidence interval of [5.1%, 8.5%] that sits well above zero. 
This confirms that targeted messaging recovers a meaningful share of users who did not organically 
adopt the feature.

## Stage 3: Premium Conversion

**Hypothesis:** Among users who adopted the feature, the targeted Premium upgrade prompt drives 
higher free-to-paid conversion in the treatment group than control.

This stage represents the primary business outcome of the experiment. Both a chi-square test and 
an independent samples t-test were applied to provide converging frequentist evidence, alongside 
a Bayesian posterior comparison.

In [6]:
conversion = results["conversion"]

print(f"Control conversion rate:    {conversion['control_rate']:.1%}")
print(f"Treatment conversion rate:  {conversion['treatment_rate']:.1%}")
print(f"Observed lift:              {conversion['observed_lift']:.1%}")
print(f"95% CI on lift:             [{conversion['ci_low']:.1%}, {conversion['ci_high']:.1%}]")
print(f"Chi-square statistic:       {conversion['chi2_statistic']}")
print(f"P-value:                    {conversion['p_value']}")
print(f"T-statistic:                {conversion['t_statistic']}")
print(f"Significant:                {conversion['significant']}")
print(f"P(treatment > control):     {conversion['prob_treatment_wins']:.1%}")

Control conversion rate:    0.4%
Treatment conversion rate:  4.3%
Observed lift:              4.0%
95% CI on lift:             [3.2%, 4.7%]
Chi-square statistic:       171.8642
P-value:                    0.0
T-statistic:                -13.291
Significant:                True
P(treatment > control):     100.0%


### Findings

The treatment group converted to Premium at 4.3% compared to 0.4% in control — a 4.0 percentage 
point lift. Both the chi-square test and the independent samples t-test return p-values of 
effectively zero, and the confidence interval of [3.2%, 4.7%] confirms the effect is real and 
consistent. The Bayesian analysis assigns 100% probability to treatment outperforming control. 
The convergence of two independent frequentist tests alongside the Bayesian result makes this 
the strongest and most defensible finding in the experiment.

## Multiple Testing Correction

Running three simultaneous hypothesis tests inflates the family-wise error rate above the nominal 
5% threshold. Bonferroni correction controls this by dividing alpha by the number of tests, 
applying a stricter significance threshold of 0.0167 to each individual test.

In [7]:
bonferroni = results["bonferroni"]

print(f"Adjusted alpha:     {bonferroni['adjusted_alpha']}")
print(f"Original p-values:  {bonferroni['original_p_values']}")
print(f"Significant:        {bonferroni['significant']}")

Adjusted alpha:     0.0167
Original p-values:  [0.0, 0.0, 0.0]
Significant:        [True, True, True]


### Findings

All three p-values remain effectively zero after applying the Bonferroni correction. The stricter 
threshold of 0.0167 does not change any conclusions — every stage of the funnel survives the 
correction. This confirms that the results are not artifacts of multiple comparisons.

## Conclusion

The experiment provides statistically significant evidence across all three funnel stages that 
the AI playlist feature drives meaningful improvements in adoption, re-engagement, and Premium 
conversion.

| Stage | Control | Treatment | Lift | Significant |
|---|---|---|---|---|
| Feature Adoption | 5.3% | 25.7% | +20.4pp | Yes |
| Notification Re-engagement | 7.5% | 14.3% | +6.8pp | Yes |
| Premium Conversion | 0.4% | 4.3% | +4.0pp | Yes |

All findings survive Bonferroni correction for multiple comparisons. Bayesian analysis assigns 
100% posterior probability to treatment outperforming control at every stage. The convergence 
of chi-square tests, a t-test, and Bayesian inference across all three stages provides strong 
and consistent evidence in favor of a full feature launch.

**Recommendation:** Ship the feature.